# ENG50 — ODE Demo Notebook
**Introduction to Engineering · Module 4 — Differential Equations in Python**

An **Ordinary Differential Equation (ODE)** describes how something *changes over time*.

> dy/dt = f(t, y)

- Left side: the **rate of change** of y
- Right side: a formula telling us *how fast* y changes at each moment

We use Python's `solve_ivp` to solve these numerically — no calculus by hand required.

In [ ]:
from scipy.integrate import solve_ivp
import numpy as np
import matplotlib.pyplot as plt
print("Libraries loaded!")

---
## Example 1: Exponential Decay ☢️

Imagine a radioactive substance slowly disappearing. The more you have, the faster it decays.

**Where the equation comes from — the solution:**

$$N(t) = N_0 \cdot e^{-kt}$$

Differentiate with respect to t (chain rule — bring the −k down from the exponent):

$$\frac{dN}{dt} = N_0 \cdot (-k) \cdot e^{-kt} = -k \cdot N$$

**The ODE:**

$$\frac{dN}{dt} = -k \cdot N$$

Where:
- `N` = amount of substance remaining
- `k` = decay rate (bigger k = decays faster)
- The **minus sign** means N is getting smaller over time

**Key idea:** right-hand side = rate of change. Negative right side → quantity is decreasing.

In [ ]:
# ── Step 1: Define constants ──────────────────────────
k  = 0.3   # decay rate (1/sec)
N0 = 100   # starting amount

# ── Step 2: Define the ODE function ───────────────────
def decay(t, y):
    # t = current time (passed by solve_ivp; not used here)
    # y = current state: y[0] is N
    return [-k * y[0]]   # dN/dt = -k * N

# ── Step 3: Solve ─────────────────────────────────────
t_span = [0, 20]
y0     = [N0]
t_eval = np.linspace(0, 20, 200)

sol = solve_ivp(decay, t_span, y0, t_eval=t_eval)

# ── Step 4: Plot ──────────────────────────────────────
# sol.t      → 1D array of time values:  [0, 0.1, 0.2, ..., 20]
# sol.y[0]   → 1D array of N values:     [100, 97, 94.1, ..., 0.2]
#              [0] because N is the 1st (only) variable we're solving
plt.plot(sol.t, sol.y[0], 'b-', label='N(t)')
plt.xlabel('Time (s)')
plt.ylabel('Amount remaining')
plt.title('Exponential Decay')
plt.grid(True)
plt.legend()
plt.show()

# Check a specific time using argmin trick
idx = np.argmin(np.abs(sol.t - 10))   # index closest to t=10
print(f"Amount left at t=10: {sol.y[0][idx]:.1f}")

# Find first time N drops below 10
idx2 = np.argmax(sol.y[0] < 10)
print(f"Drops below 10 at t = {sol.t[idx2]:.1f} s")

### 🔍 Deep Dive — How solve_ivp works internally

**The 4 arguments:**
```python
solve_ivp(decay,   t_span,  y0,      t_eval=t_eval)
#         ^        ^         ^         ^
#         ODE fn   [0,20]   [100]     output times
```

**The stepping math:**
solve_ivp uses tiny time steps. At each step:

> N_new = N_old + rate × Δt

Example with Δt = 0.01:
- t=0.00: N=100.0, rate = −0.3×100 = −30.0 → N_new = 100 + (−30)×0.01 = **99.7**
- t=0.01: N=99.7,  rate = −0.3×99.7 = −29.9 → N_new = 99.7 + (−29.9)×0.01 = **99.4**

**Why sol.y is 2D:**
solve_ivp is built for systems with multiple variables. Even with one variable, sol.y has one *row* per variable:
- sol.y[0] = N values (row 0)
- sol.y[1] = second variable (if you had one)

**Two index tricks:**
- `np.argmin(np.abs(sol.t - 10))` → index of time closest to t=10
- `np.argmax(sol.y[0] < 10)` → index of first time N drops below 10

### ✏️ Your Turn — Example 1

1. Change `k = 0.1`. How does the curve look different?
2. Change `N0 = 500`. Does the shape change, or just the scale?
3. Find the time when N first drops below 10. Use `np.argmax(sol.y[0] < 10)`.

---
## Example 2: Newton's Law of Cooling ☕

A cup of coffee (90°C) in a 20°C room. How fast does it cool?

Newton's Law: **the rate of cooling depends on how much hotter the object is than its surroundings.**

**Where the equation comes from — the solution:**

$$T(t) = T_\infty + (T_0 - T_\infty) \cdot e^{-kt}$$

Differentiate with respect to t — T∞ and T₀ are constants, only e^(−kt) changes:

$$\frac{dT}{dt} = (T_0 - T_\infty) \cdot (-k) \cdot e^{-kt} = -k \cdot (T - T_\infty)$$

**The ODE:**

$$\frac{dT}{dt} = -k \cdot (T - T_\infty)$$

Where:
- `T` = current temperature
- `T∞` = room temperature (the target)
- `k` = cooling rate

In [ ]:
# ── Constants ─────────────────────────────────────────
k    = 0.08  # cooling rate [1/min]
Tinf = 20    # room temperature [°C]
T0   = 90    # initial coffee temperature [°C]

# ── ODE ───────────────────────────────────────────────
def cooling(t, y):
    return [-k * (y[0] - Tinf)]   # dT/dt = -k*(T - Tinf)

# ── Solve ─────────────────────────────────────────────
t_eval = np.linspace(0, 60, 300)
sol    = solve_ivp(cooling, [0, 60], [T0], t_eval=t_eval)

# ── Plot ──────────────────────────────────────────────
# sol.t      → time array:  [0, 0.2, 0.4, ..., 60]  (minutes)
# sol.y[0]   → temp array:  [90, 87.1, 84.3, ..., 20.4]  (°C)
#              [0] = first variable = T (temperature)
plt.plot(sol.t, sol.y[0], 'r-', label='Coffee temp')
plt.axhline(Tinf, color='gray', linestyle='--', label=f'Room temp = {Tinf}°C')
plt.xlabel('Time (min)')
plt.ylabel('Temperature (°C)')
plt.title("Newton's Law of Cooling")
plt.grid(True)
plt.legend()
plt.show()

# When does coffee get within 1°C of room temperature?
idx = np.argmax(np.abs(sol.y[0] - Tinf) < 1)
print(f"Within 1°C of room temp at t = {sol.t[idx]:.1f} min")

### ✏️ Your Turn — Example 2

1. Change `Tinf = 5` (a cold fridge). Does the coffee cool faster or slower?
2. Change `T0 = 60`. How long until the coffee is within 1°C of room temp?
3. Try `T0 = 10, Tinf = 25` (warming up). Which direction does the curve go?

---
## Example 3: RC Circuit (Charging Capacitor) ⚡

A battery (Vs) charges a capacitor through a resistor. As it charges, charging slows down.

**Where the equation comes from — the solution:**

$$Vc(t) = Vs \cdot (1 - e^{-t/RC})$$

Differentiate with respect to t (chain rule on e^(−t/RC), coefficient is −1/RC):

$$\frac{dVc}{dt} = Vs \cdot e^{-t/RC} \cdot \frac{1}{RC} = \frac{Vs - Vc}{RC}$$

**The ODE:**

$$\frac{dVc}{dt} = \frac{Vs - Vc}{R \cdot C}$$

**Time constant τ = RC:** at t = τ, the capacitor reaches 63.2% of Vs. Always.

In [ ]:
# ── Circuit parameters ────────────────────────────────
R  = 1000    # Resistance: 1000 Ohms (1kΩ)
C  = 0.0001  # Capacitance: 100 microfarads
Vs = 5.0     # Supply voltage [V]
Vc0 = 0.0   # Initial capacitor voltage [V]
tau = R * C  # Time constant [seconds]
print(f"Time constant τ = {tau:.4f} s")

# ── ODE ───────────────────────────────────────────────
def rc_circuit(t, y):
    return [(Vs - y[0]) / (R * C)]   # dVc/dt = (Vs - Vc) / RC

# ── Solve ─────────────────────────────────────────────
t_end  = 5 * tau
t_eval = np.linspace(0, t_end, 300)
sol    = solve_ivp(rc_circuit, [0, t_end], [Vc0], t_eval=t_eval)

# ── Plot ──────────────────────────────────────────────
# sol.t      → time array:  [0, 0.0017, ..., 0.5]  (seconds)
# sol.y[0]   → voltage array: [0, 0.63, ..., 4.97]  (volts)
#              [0] = first variable = Vc (capacitor voltage)
plt.plot(sol.t, sol.y[0], 'g-', label='Capacitor Vc')
plt.axhline(Vs, color='blue', linestyle='--', label=f'Supply Vs = {Vs}V')
plt.axvline(tau, color='orange', linestyle=':', label=f'1τ = {tau:.4f}s')
plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')
plt.title('RC Circuit — Capacitor Charging')
plt.grid(True)
plt.legend()
plt.show()

# Voltage at exactly 1 tau
idx = np.argmin(np.abs(sol.t - tau))
print(f"Voltage at t=τ: {sol.y[0][idx]:.3f} V  ({sol.y[0][idx]/Vs*100:.1f}% of Vs)")
print(f"Final voltage:  {sol.y[0][-1]:.3f} V")

### ✏️ Your Turn — Example 3

1. Change `R = 2000`. How does τ change? Does the circuit charge faster or slower?
2. Change `Vs = 12`. What is the final voltage? Use `sol.y[0][-1]`
3. Verify the 63.2% rule at t=τ using the argmin trick.

---
## Example 4: Logistic Population Growth 🐟

Fish in a lake multiply freely at first, but slow as the lake fills up.

**Where the equation comes from — the solution:**

$$N(t) = \frac{K}{1 + \frac{K - N_0}{N_0} \cdot e^{-rt}}$$

Differentiating this (messy!) eventually simplifies to:

$$\frac{dN}{dt} = r \cdot N \cdot \left(1 - \frac{N}{K}\right)$$

This is why `solve_ivp` is so useful — the solution formula is complicated, but the ODE itself is clean.

Where:
- `r` = growth rate
- `K` = carrying capacity (maximum the environment supports)
- `(1 − N/K)` = the "brake" — goes to 0 as N approaches K

In [ ]:
# ── Parameters ────────────────────────────────────────
r  = 0.5     # growth rate (per year)
K  = 1000    # carrying capacity (max fish)
N0 = 50      # starting population

# ── ODE ───────────────────────────────────────────────
def logistic(t, y):
    N = y[0]
    return [r * N * (1 - N / K)]   # dN/dt = r*N*(1 - N/K)

# ── Solve ─────────────────────────────────────────────
t_eval = np.linspace(0, 20, 300)
sol    = solve_ivp(logistic, [0, 20], [N0], t_eval=t_eval)

# ── Plot ──────────────────────────────────────────────
# sol.t      → time array:  [0, 0.067, ..., 20]  (years)
# sol.y[0]   → population:  [50, 51.7, ..., 999.8]  (fish)
#              [0] = first variable = N (population size)
plt.plot(sol.t, sol.y[0], color='purple', label='Fish population')
plt.axhline(K,   color='red',  linestyle='--', label=f'Capacity K = {K}')
plt.axhline(K/2, color='gray', linestyle=':', alpha=0.5, label='K/2 (fastest growth)')
plt.xlabel('Time (years)')
plt.ylabel('Population')
plt.title('Logistic Population Growth')
plt.grid(True)
plt.legend()
plt.show()

print(f"Final population: {sol.y[0][-1]:.0f} fish")

### ✏️ Your Turn — Example 4

1. Change `N0 = 500` (start at half capacity). Does the S-curve disappear?
2. Change `N0 = 1200` (start above capacity). Which direction does population move?
3. Change `r = 1.5` (faster fish). How much sooner does population reach capacity?

---
## Common Mistakes

| Mistake | Fix |
|---|---|
| `y0 = 100` | Must be a list: `y0 = [100]` |
| `solve_ivp(decay(), ...)` | No parentheses: `solve_ivp(decay, ...)` |
| `sol.y` instead of `sol.y[0]` | Need the first row: `sol.y[0]` |
| Forgetting `t` in ODE function | Always `def ode(t, y):` even if t unused |
| `t_span` and `t_eval` mismatch | `t_eval` must be within `t_span` |

---
## Quick Reference Template

```python
from scipy.integrate import solve_ivp
import numpy as np
import matplotlib.pyplot as plt

# 1. Constants
k = 0.3

# 2. ODE function  (t and y ALWAYS required)
def my_ode(t, y):
    return [-k * y[0]]   # dy/dt = ...

# 3. Solve
t_eval = np.linspace(0, 20, 300)
sol = solve_ivp(my_ode, [0, 20], [100], t_eval=t_eval)

# 4. Plot
plt.plot(sol.t, sol.y[0])
plt.xlabel('Time'); plt.ylabel('Value')
plt.title('My ODE Solution')
plt.grid(True); plt.show()
```

**solve_ivp arguments:**

| argument | meaning |
|---|---|
| `my_ode` | the ODE function (no parentheses) |
| `[0, 20]` | time range [start, end] |
| `[100]` | initial condition (must be a list) |
| `t_eval` | times you want in the output (optional, for smooth plots) |

---
## Final Challenge — Design Your Own

Using the template above, model one of these:

1. **Medicine:** A drug dose of 200 mg eliminated at rate k=0.1/hr. How long until < 10 mg remains?
2. **Bank account:** $5000 earning continuous interest at 3%/year (k=0.03). Balance after 20 years? (dM/dt = k·M)
3. **Your own idea:** Think of anything that changes over time and write its ODE!